In [43]:
from pathlib import Path
import pandas as pd
import numpy as np

BRONZE_DIR = Path("../data")
SILVER_DIR = Path("../data/silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

def read_csv_safe(filename):
    path = BRONZE_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    print(f"Lido: {filename} | shape={df.shape}")
    return df

def save_silver(df, filename):
    out = SILVER_DIR / filename
    df.to_csv(out, index=False)
    print(f"Arquivo salvo em: {out} | linhas: {len(df):,}")

def show_quality(df, nome):
    print(f"\n--- {nome} ---")
    print("shape:", df.shape)
    print("duplicadas:", df.duplicated().sum())
    print("nulos por coluna:")
    print(df.isnull().sum().sort_values(ascending=False))

def normalize_text_cols(df):
    for col in df.select_dtypes(include="object").columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .replace({"": np.nan, "None": np.nan, "nan": np.nan})
        )
    return df

In [44]:
def normalize_city(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    )


def normalize_state(series):
    return (
        series.astype(str)
        .str.strip()
        .str.upper()
        .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    )


def normalize_zip(series):
    s = series.astype(str).str.extract(r"(\d+)", expand=False)
    return s.str.zfill(5)


def to_datetime_cols(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


def remove_empty_strings(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = df[col].replace(r"^\s*$", pd.NA, regex=True)
    return df

In [45]:
customers = read_csv_safe("olist_customers_dataset.csv")
show_quality(customers, "customers raw")

customers["customer_city"] = normalize_city(customers["customer_city"])
customers["customer_state"] = normalize_state(customers["customer_state"])
customers["customer_zip_code_prefix"] = normalize_zip(customers["customer_zip_code_prefix"])

customers = remove_empty_strings(
    customers,
    ["customer_unique_id", "customer_city", "customer_state"]
)

customers = customers.dropna(subset=["customer_id", "customer_unique_id"])

customers = customers.drop_duplicates(subset=["customer_id"]).copy()

assert customers["customer_state"].str.len().le(2).all(), "Estado inválido!"
assert customers["customer_zip_code_prefix"].str.len().eq(5).all(), "ZIP inválido!"

show_quality(customers, "customers silver")
save_silver(customers, "olist_customers_dataset.csv")

Lido: olist_customers_dataset.csv | shape=(99441, 5)

--- customers raw ---
shape: (99441, 5)
duplicadas: 0
nulos por coluna:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

--- customers silver ---
shape: (99441, 5)
duplicadas: 0
nulos por coluna:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
Arquivo salvo em: ..\data\silver\olist_customers_dataset.csv | linhas: 99,441


In [46]:
sellers = read_csv_safe("olist_sellers_dataset.csv")
show_quality(sellers, "sellers raw")

sellers["seller_city"] = normalize_city(sellers["seller_city"])
sellers["seller_state"] = normalize_state(sellers["seller_state"])
sellers["seller_zip_code_prefix"] = normalize_zip(sellers["seller_zip_code_prefix"])

sellers = remove_empty_strings(sellers, ["seller_city", "seller_state"])

sellers = sellers.dropna(subset=["seller_id"])

sellers = sellers.drop_duplicates(subset=["seller_id"]).copy()

show_quality(sellers, "sellers silver")
save_silver(sellers, "olist_sellers_dataset.csv")

Lido: olist_sellers_dataset.csv | shape=(3095, 4)

--- sellers raw ---
shape: (3095, 4)
duplicadas: 0
nulos por coluna:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

--- sellers silver ---
shape: (3095, 4)
duplicadas: 0
nulos por coluna:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64
Arquivo salvo em: ..\data\silver\olist_sellers_dataset.csv | linhas: 3,095


In [48]:
cat = read_csv_safe("product_category_name_translation.csv")
show_quality(cat, "category translation raw")

cat["product_category_name"] = (
    cat["product_category_name"].astype(str).str.strip().str.lower()
)

cat["product_category_name_english"] = (
    cat["product_category_name_english"].astype(str).str.strip().str.lower()
)

cat = cat.dropna(subset=["product_category_name", "product_category_name_english"])
cat = cat.drop_duplicates(subset=["product_category_name"]).copy()

if "desconhecida" not in cat["product_category_name"].values:
    cat = pd.concat([cat, pd.DataFrame([["desconhecida", "unknown"]], columns=cat.columns)], ignore_index=True)

show_quality(cat, "category translation silver")
save_silver(cat, "product_category_name_translation.csv")

Lido: product_category_name_translation.csv | shape=(71, 2)

--- category translation raw ---
shape: (71, 2)
duplicadas: 0
nulos por coluna:
product_category_name            0
product_category_name_english    0
dtype: int64

--- category translation silver ---
shape: (72, 2)
duplicadas: 0
nulos por coluna:
product_category_name            0
product_category_name_english    0
dtype: int64
Arquivo salvo em: ..\data\silver\product_category_name_translation.csv | linhas: 72


In [50]:
products = read_csv_safe("olist_products_dataset.csv")
show_quality(products, "products raw")

missing_before = products.isnull().mean().sort_values(ascending=False)

products["product_category_name"] = (
    products["product_category_name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"nan": "desconhecida"})
)

products = products.drop_duplicates(subset=["product_id"]).copy()

num_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in num_cols:
    if col in products.columns:
        products[col] = pd.to_numeric(products[col], errors="coerce")

dim_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in dim_cols:
    if col in products.columns:
        products[col] = products[col].fillna(products[col].median())

products = products[
    (products["product_weight_g"] > 0) & (products["product_weight_g"] <= 30000) &
    (products["product_length_cm"] > 0) & (products["product_length_cm"] <= 200) &
    (products["product_height_cm"] > 0) & (products["product_height_cm"] <= 200) &
    (products["product_width_cm"] > 0) & (products["product_width_cm"] <= 200)
].copy()

products["product_photos_qty"] = products["product_photos_qty"].fillna(0)
products["product_name_lenght"] = products["product_name_lenght"].fillna(products["product_name_lenght"].median())
products["product_description_lenght"] = products["product_description_lenght"].fillna(products["product_description_lenght"].median())

float_cols = products.select_dtypes(include=["float"]).columns
products[float_cols] = products[float_cols].round(2)

missing_after = products.isnull().mean().sort_values(ascending=False)

quality_score = (1 - missing_after.mean()) * 100

show_quality(products, "products silver")
print("missing_before:\n", missing_before)
print("missing_after:\n", missing_after)
print("quality_score:", round(quality_score, 2))

save_silver(products, "olist_products_dataset.csv")

Lido: olist_products_dataset.csv | shape=(32951, 9)

--- products raw ---
shape: (32951, 9)
duplicadas: 0
nulos por coluna:
product_category_name         610
product_description_lenght    610
product_name_lenght           610
product_photos_qty            610
product_weight_g                2
product_height_cm               2
product_length_cm               2
product_width_cm                2
product_id                      0
dtype: int64

--- products silver ---
shape: (32946, 9)
duplicadas: 0
nulos por coluna:
product_category_name         610
product_id                      0
product_name_lenght             0
product_description_lenght      0
product_photos_qty              0
product_weight_g                0
product_length_cm               0
product_height_cm               0
product_width_cm                0
dtype: int64
missing_before:
 product_category_name         0.018512
product_description_lenght    0.018512
product_name_lenght           0.018512
product_photos_qty           

In [51]:
products = products.dropna(subset=[
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]).copy()

products["product_category_name"] = products["product_category_name"].fillna("desconhecida")

products["product_photos_qty"] = products["product_photos_qty"].fillna(0)

products["product_name_lenght"] = products["product_name_lenght"].fillna(
    products["product_name_lenght"].median()
)

products["product_description_lenght"] = products["product_description_lenght"].fillna(
    products["product_description_lenght"].median()
)

In [52]:
orders = read_csv_safe("olist_orders_dataset.csv")
show_quality(orders, "orders raw")

orders = orders.drop_duplicates(subset=["order_id"]).copy()

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders = to_datetime_cols(orders, date_cols)

orders = orders.dropna(subset=["order_id", "customer_id", "order_purchase_timestamp"])

orders["order_status"] = orders["order_status"].astype(str).str.strip().str.lower()

orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

orders["is_delivered"] = orders["order_delivered_customer_date"].notna()

orders["is_late"] = (
    orders["delivery_delay_days"].notna() &
    (orders["delivery_delay_days"] > 0)
)

orders = orders[
    orders["order_approved_at"].isna() |
    (orders["order_approved_at"] >= orders["order_purchase_timestamp"])
].copy()

show_quality(orders, "orders silver")
save_silver(orders, "olist_orders_dataset.csv")

Lido: olist_orders_dataset.csv | shape=(99441, 8)

--- orders raw ---
shape: (99441, 8)
duplicadas: 0
nulos por coluna:
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
order_purchase_timestamp            0
order_status                        0
customer_id                         0
order_estimated_delivery_date       0
dtype: int64

--- orders silver ---
shape: (99441, 12)
duplicadas: 0
nulos por coluna:
delivery_days                    2965
delivery_delay_days              2965
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_status                        0
customer_id                         0
order_id                            0
order_purchase_timestamp            0
order_estimated_delivery_date       0
is_delivered                        0
is_late                             0
dtype: int64
Arquivo salvo em: ..

In [53]:
items = read_csv_safe("olist_order_items_dataset.csv")
show_quality(items, "order items raw")

items = items.drop_duplicates(subset=["order_id", "order_item_id"]).copy()

items["price"] = pd.to_numeric(items["price"], errors="coerce")
items["freight_value"] = pd.to_numeric(items["freight_value"], errors="coerce")
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")

items = items.dropna(subset=["order_id", "order_item_id", "product_id", "seller_id", "price"])
items = items[items["price"] > 0].copy()

items["freight_value"] = items["freight_value"].fillna(items["freight_value"].median()).clip(lower=0)

items["total_item_value"] = items["price"] + items["freight_value"]

p999 = items["price"].quantile(0.999)

items["is_price_outlier"] = items["price"] > p999

items = items[~items["is_price_outlier"]].copy()

items["price"] = items["price"].round(2)
items["freight_value"] = items["freight_value"].round(2)
items["total_item_value"] = items["total_item_value"].round(2)

show_quality(items, "order items silver")
save_silver(items, "olist_order_items_dataset.csv")

Lido: olist_order_items_dataset.csv | shape=(112650, 7)

--- order items raw ---
shape: (112650, 7)
duplicadas: 0
nulos por coluna:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

--- order items silver ---
shape: (112545, 9)
duplicadas: 0
nulos por coluna:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
total_item_value       0
is_price_outlier       0
dtype: int64
Arquivo salvo em: ..\data\silver\olist_order_items_dataset.csv | linhas: 112,545


In [54]:
payments = read_csv_safe("olist_order_payments_dataset.csv")
show_quality(payments, "payments raw")

payments["payment_type"] = (
    payments["payment_type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

payments = payments[payments["payment_type"] != "not_defined"].copy()

payments["payment_sequential"] = pd.to_numeric(payments["payment_sequential"], errors="coerce").astype("Int64")
payments["payment_installments"] = pd.to_numeric(payments["payment_installments"], errors="coerce")
payments["payment_value"] = pd.to_numeric(payments["payment_value"], errors="coerce")

payments = payments.dropna(subset=[
    "order_id",
    "payment_sequential",
    "payment_type",
    "payment_value"
])

payments = payments[payments["payment_value"] > 0].copy()

payments["payment_installments"] = (
    payments["payment_installments"]
    .fillna(payments["payment_installments"].median())
    .clip(lower=1)
    .round(0)
    .astype("Int64")
)

payments = payments.drop_duplicates(subset=["order_id", "payment_sequential"]).copy()

float_cols = payments.select_dtypes(include=["float"]).columns
payments[float_cols] = payments[float_cols].round(2)

show_quality(payments, "payments silver")
save_silver(payments, "olist_order_payments_dataset.csv")

Lido: olist_order_payments_dataset.csv | shape=(103886, 5)

--- payments raw ---
shape: (103886, 5)
duplicadas: 0
nulos por coluna:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

--- payments silver ---
shape: (103877, 5)
duplicadas: 0
nulos por coluna:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
Arquivo salvo em: ..\data\silver\olist_order_payments_dataset.csv | linhas: 103,877


In [55]:
reviews = read_csv_safe("olist_order_reviews_dataset.csv")
show_quality(reviews, "reviews raw")

reviews = reviews.drop_duplicates(subset=["review_id"]).copy()

reviews["review_score"] = pd.to_numeric(reviews["review_score"], errors="coerce")
reviews = reviews[reviews["review_score"].between(1, 5)].copy()

date_cols = [
    "review_creation_date",
    "review_answer_timestamp"
]

reviews = to_datetime_cols(reviews, date_cols)

reviews["review_comment_title"] = reviews["review_comment_title"].fillna(pd.NA)
reviews["review_comment_message"] = reviews["review_comment_message"].fillna(pd.NA)

reviews["has_comment"] = (
    reviews["review_comment_message"]
    .astype(str)
    .str.strip()
    .ne("")
)

reviews["sentiment"] = reviews["review_score"].map({
    1: "muito negativo",
    2: "negativo",
    3: "neutro",
    4: "positivo",
    5: "muito positivo"
})

show_quality(reviews, "reviews silver")
save_silver(reviews, "olist_order_reviews_dataset.csv")

Lido: olist_order_reviews_dataset.csv | shape=(99224, 7)

--- reviews raw ---
shape: (99224, 7)
duplicadas: 0
nulos por coluna:
review_comment_title       87656
review_comment_message     58247
review_id                      0
review_score                   0
order_id                       0
review_creation_date           0
review_answer_timestamp        0
dtype: int64

--- reviews silver ---
shape: (98410, 9)
duplicadas: 0
nulos por coluna:
review_comment_title       86891
review_comment_message     57742
review_id                      0
review_score                   0
order_id                       0
review_creation_date           0
review_answer_timestamp        0
has_comment                    0
sentiment                      0
dtype: int64
Arquivo salvo em: ..\data\silver\olist_order_reviews_dataset.csv | linhas: 98,410


In [41]:
geo = read_csv_safe("olist_geolocation_dataset.csv")
show_quality(geo, "geolocation raw")

geo["geolocation_lat"] = pd.to_numeric(geo["geolocation_lat"], errors="coerce")
geo["geolocation_lng"] = pd.to_numeric(geo["geolocation_lng"], errors="coerce")

geo = geo.dropna(subset=["geolocation_lat", "geolocation_lng"])

geo = geo[
    geo["geolocation_lat"].between(-33.8, 5.3) &
    geo["geolocation_lng"].between(-73.9, -34.7)
].copy()

geo["geolocation_city"] = normalize_city(geo["geolocation_city"])
geo["geolocation_state"] = normalize_state(geo["geolocation_state"])
geo["geolocation_zip_code_prefix"] = normalize_zip(geo["geolocation_zip_code_prefix"])

geo = geo[geo["geolocation_zip_code_prefix"].notna()]

geo[["geolocation_lat", "geolocation_lng"]] = geo[
    ["geolocation_lat", "geolocation_lng"]
].round(6)

show_quality(geo, "geolocation silver")
save_silver(geo, "olist_geolocation_dataset.csv")


--- geolocation raw ---
shape: (1000163, 5)
duplicadas: 261831
nulos por coluna:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

--- geolocation silver ---
shape: (1000121, 5)
duplicadas: 262001
nulos por coluna:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
Arquivo salvo em: ..\data\silver\olist_geolocation_dataset.csv | linhas: 1,000,121


In [42]:
import os

print("Arquivos gerados na camada silver:")
for file in sorted(os.listdir(SILVER_DIR)):
    print("-", file)

Arquivos gerados na camada silver:
- olist_customers_dataset.csv
- olist_geolocation_dataset.csv
- olist_order_items_dataset.csv
- olist_order_payments_dataset.csv
- olist_order_reviews_dataset.csv
- olist_orders_dataset.csv
- olist_products_dataset.csv
- olist_sellers_dataset.csv
- product_category_name_translation.csv
